## Elección del modelo de clasificación🔬 

En este notebook testeo posibles algoritmos de clasificación una vez obtenidas las caras vectorizadas y con sus respectivos labels de emoción.

Pruebo distintos modelos...

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import (
    cross_val_score,
    GridSearchCV
)

from sklearn.neighbors import KNeighborsClassifier

In [53]:
EMOTION_CATEGORIES = ['happy', 'sad', 'neutral']

In [57]:
def _load_dataset(faces_path: str) -> pd.DataFrame:  

    emotions_df = {}
    for emotion in EMOTION_CATEGORIES:
        df = pd.read_csv(f"{faces_path}/{emotion}_faces.csv", header=None).values
        size = df.shape[0]
        emotions_df[emotion] = {
            'features': df,
            'size': size
        }

    faces_features = np.concatenate(
        [
            emotions_df[emotion]['features']
            for emotion in EMOTION_CATEGORIES
        ], 
        axis=0
    )
    faces_features = pd.DataFrame(faces_features)

    labels = np.concatenate(
        [
            np.full(emotion_data['size'], emotion_name)
            for emotion_name, emotion_data in emotions_df.items()
        ], 
        axis=0
    )

    if faces_features[faces_features.columns[-1]].dtype != object:
        faces_features['label'] = labels
    else:
        faces_features.rename(columns={faces_features.columns[-1]: 'label'}, inplace=True)

    return faces_features

pca_faces = _load_dataset("../data/processed/pca_faces")

In [17]:
print(pca_faces.groupby('label').size().reset_index(name='count').to_markdown(index=False))

| label   |   count |
|:--------|--------:|
| happy   |    3884 |
| neutral |    2775 |
| sad     |    1545 |


In [33]:
Y = pca_faces['label']
X = pca_faces.copy().drop(columns=['label'])

### Nested Cross Validation

In [39]:
param_grid = {
    'n_neighbors': [3, 5, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'cosine'],
}

grid_search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5
)

scores = cross_val_score(grid_search, X, Y, cv=4)

In [40]:
scores.mean()

np.float64(0.7322038030229157)

In [48]:
grid_search.fit(X, Y)

,estimator,KNeighborsClassifier()
,param_grid,"{'metric': ['euclidean', 'cosine'], 'n_neighbors': [3, 5, ...], 'weights': ['uniform', 'distance']}"
,scoring,None
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_neighbors,11


In [52]:
results = pd.DataFrame(grid_search.cv_results_)
results.sort_values('rank_test_score').head(8)[
    ['param_metric', 'param_weights', 'param_n_neighbors', 'mean_test_score']
]

,param_metric,param_weights,param_n_neighbors,mean_test_score
11,cosine,distance,11,0.735860
9,cosine,distance,5,0.727937
5,euclidean,distance,11,0.727084
3,euclidean,distance,5,0.721111
7,cosine,distance,3,0.711482
1,euclidean,distance,3,0.709896
10,cosine,uniform,11,0.707825
8,cosine,uniform,5,0.704289


### Comparación justa: Raw Pixels + PCA vs LBP + PCA vs HOG + PCA

Comparamos los 3 descriptores bajo las mismas condiciones para que la diferencia de accuracy refleje al descriptor y no otra cosa:

- **Mismas imágenes** para los 3 (extraídas una sola vez, en el mismo orden, del mismo subset de `data/dataset/fer2013`).
- **Mismo preprocesamiento**: gris + `equalizeHist` (igual que `FaceClassifier::vectorizeFace` en `classifier.cpp`).
- **LBP por bloques** (no un histograma global): se calcula LBP celda por celda (`8x8`, igual que las celdas de HOG) y se concatenan los histogramas, así conserva estructura espacial en vez de perderla toda en un único histograma global.
- **Mismo pipeline downstream**: centrar (restar la media) → PCA a 100 componentes (=`PCA_DIM`) → mismo `GridSearchCV` con el mismo `param_grid` que se usó para `HOG + PCA` arriba.

In [ ]:
import cv2
from pathlib import Path
from skimage.feature import local_binary_pattern

FACES_DIR = "../data/dataset/fer2013"
MAX_PER_CLASS = None

HOG_PARAMS = dict(  # mismos valores que HOG_BLOCK_SIZE/STRIDE/CELL/NBINS en constants.h
    _winSize=(48, 48), _blockSize=(16, 16), _blockStride=(8, 8), _cellSize=(8, 8), _nbins=9
)
_hog = cv2.HOGDescriptor(*HOG_PARAMS.values())

def extract_hog(image):
    return _hog.compute(image).flatten()

def extract_blocked_lbp(image, cell_size=8, radius=1, n_points=8):
    h, w = image.shape
    feats = []
    for y in range(0, h, cell_size):
        for x in range(0, w, cell_size):
            cell = image[y:y + cell_size, x:x + cell_size]
            lbp = local_binary_pattern(cell, n_points, radius, method='uniform')
            hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, n_points + 3), density=True)
            feats.append(hist)
    return np.concatenate(feats)

X_raw_list, X_lbp_list, X_hog_list, Y_fair = [], [], [], []

for emotion in EMOTION_CATEGORIES:
    files = sorted(Path(f"{FACES_DIR}/{emotion}").glob("*.jpg"))
    if MAX_PER_CLASS is not None:
        files = files[:MAX_PER_CLASS]
    print(f"  {emotion}: {len(files)} caras")

    for fp in files:
        img = cv2.imread(str(fp), cv2.IMREAD_GRAYSCALE)  
        img = cv2.equalizeHist(img)                      

        X_raw_list.append(img.flatten() / 255.0)
        X_lbp_list.append(extract_blocked_lbp(img))
        X_hog_list.append(extract_hog(img))
        Y_fair.append(emotion)

X_raw_fair = np.array(X_raw_list)
X_lbp_fair = np.array(X_lbp_list)
X_hog_fair = np.array(X_hog_list)
Y_fair = np.array(Y_fair)

print(f"\nDimensiones (mismas {len(Y_fair)} imágenes para las 3):")
print(f"  Raw pixels: {X_raw_fair.shape}")
print(f"  LBP (bloqueado): {X_lbp_fair.shape}")
print(f"  HOG: {X_hog_fair.shape}")

  happy: 8989 caras
  sad: 6077 caras
  neutral: 6198 caras

Dimensiones (mismas 21264 imágenes para las 3):
  Raw pixels: (21264, 2304)
  LBP (bloqueado): (21264, 360)
  HOG: (21264, 900)


In [66]:
from sklearn.decomposition import PCA

def evaluate_with_pca(X_feat, y, n_components=100, name=""):
    """Centra, proyecta a PCA_DIM=100 y corre el mismo grid search que HOG+PCA (param_grid de la celda de arriba)."""
    mean_vec = X_feat.mean(axis=0)
    X_centered = X_feat - mean_vec

    n_comp = min(n_components, X_centered.shape[0], X_centered.shape[1])
    pca = PCA(n_components=n_comp, random_state=42)
    X_pca = pca.fit_transform(X_centered)

    grid = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5)
    grid.fit(X_pca, y)

    return grid

grid_raw_fair = evaluate_with_pca(X_raw_fair, Y_fair, name="Raw pixels + PCA")
grid_lbp_fair = evaluate_with_pca(X_lbp_fair, Y_fair, name="LBP (bloqueado) + PCA")
grid_hog_fair = evaluate_with_pca(X_hog_fair, Y_fair, name="HOG + PCA")

comparison_fair_df = pd.DataFrame([
    {
        'Descriptor': 'Raw pixels + PCA', 
        'Accuracy': grid_raw_fair.best_score_, 
        **grid_raw_fair.best_params_
    },
    {
        'Descriptor': 'LBP (bloqueado) + PCA', 
        'Accuracy': grid_lbp_fair.best_score_, 
        **grid_lbp_fair.best_params_
    },
    {
        'Descriptor': 'HOG + PCA', 
        'Accuracy': grid_hog_fair.best_score_, 
        **grid_hog_fair.best_params_
    },
]).sort_values('Accuracy', ascending=False)

display(comparison_fair_df)

,Descriptor,Accuracy,metric,n_neighbors,weights
2,HOG + PCA,0.676730,cosine,11,distance
0,Raw pixels + PCA,0.566027,cosine,11,distance
1,LBP (bloqueado) + PCA,0.507007,cosine,11,distance
